# Xử lý ngôn ngữ tự nhiên - CS221.Q21.KHTN

## Demo chương 17 - Sequence Labeling for Named Entity Recognition

### Nhóm 6:
- Hà Thanh Phong - 24520024
- Hà Xuân Thiện - 24520031
- Trần Quang Trường - 24521901

In [3]:
!pip install -q datasets transformers accelerate seqeval sklearn-crfsuite scikit-learn pandas torch

In [4]:
import numpy as np
import pandas as pd
from collections import Counter
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from seqeval.metrics import f1_score as entity_f1_score

# Tải bộ dữ liệu công khai WikiANN tiếng Việt cho bài toán Named Entity Recognition (NER)
print("Đang tải bộ dữ liệu WikiANN tiếng Việt...")
dataset = load_dataset("unimelb-nlp/wikiann", "vi")

# Nhãn dùng định dạng BIO, ví dụ: B-PER, I-ORG, O
label_names = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_names)

# Dùng một tập con vừa đủ để demo chạy nhanh trên Colab
train_size = min(3000, len(dataset["train"]))
test_size = min(500, len(dataset["test"]))

train_data = dataset["train"].select(range(train_size))
test_data = dataset["test"].select(range(test_size))

# Chuyển dữ liệu về dạng [(token, label), ...] giống các bài sequence labeling cổ điển
def example_to_tagged_sentence(example):
    return list(zip(example["tokens"], [label_names[tag] for tag in example["ner_tags"]]))

train_sents = [example_to_tagged_sentence(example) for example in train_data]
test_sents = [example_to_tagged_sentence(example) for example in test_data]

print(f"Label names: {label_names}")
print(f"Total training sentences available: {len(dataset['train'])}")
print(f"Total testing sentences available: {len(dataset['test'])}")
print(f"Training sentences used: {len(train_sents)}")
print(f"Testing sentences used: {len(test_sents)}")

# Quan sát phân bố lớp để thấy bài toán bị lệch nhiều về nhãn O
label_counts = Counter(label for sent in train_sents for _, label in sent)
label_distribution = pd.DataFrame({
    "Label": list(label_counts.keys()),
    "Count": list(label_counts.values()),
}).sort_values("Count", ascending=False)

display(label_distribution)

Đang tải bộ dữ liệu WikiANN tiếng Việt...


README.md: 0.00B [00:00, ?B/s]

vi/validation-00000-of-00001.parquet:   0%|          | 0.00/568k [00:00<?, ?B/s]

vi/test-00000-of-00001.parquet:   0%|          | 0.00/571k [00:00<?, ?B/s]

vi/train-00000-of-00001.parquet:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Label names: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
Total training sentences available: 20000
Total testing sentences available: 10000
Training sentences used: 3000
Testing sentences used: 500


,Label,Count
4,O,6978
6,I-ORG,4219
1,I-LOC,2592
3,I-PER,2159
0,B-LOC,1189
5,B-ORG,1106
2,B-PER,1084


## Conditional Random Fields (CRF)

In [5]:
import sklearn_crfsuite

# CRF là mô hình sequence labeling cổ điển: dự đoán nhãn của token dựa trên đặc trưng cục bộ
# và phụ thuộc giữa các nhãn liên tiếp trong câu.
def word2features(sent, i):
    word = sent[i][0]

    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[:3]": word[:3],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
    }

    if i > 0:
        word1 = sent[i - 1][0]
        features.update({
            "-1:word.lower()": word1.lower(),
            "-1:word.istitle()": word1.istitle(),
            "-1:word.isupper()": word1.isupper(),
        })
    else:
        features["BOS"] = True

    if i < len(sent) - 1:
        word1 = sent[i + 1][0]
        features.update({
            "+1:word.lower()": word1.lower(),
            "+1:word.istitle()": word1.istitle(),
            "+1:word.isupper()": word1.isupper(),
        })
    else:
        features["EOS"] = True

    return features


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]


def sent2labels(sent):
    return [label for _, label in sent]

print("Preparing features for CRF...")
X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]
X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)

print("Training CRF Model...")
crf.fit(X_train, y_train)

print("Evaluating CRF...")
crf_y_pred_seq = crf.predict(X_test)

# Flatten lists for evaluation
flat_y_test = [label for sent in y_test for label in sent]
flat_crf_y_pred = [label for sent in crf_y_pred_seq for label in sent]

crf_accuracy = accuracy_score(flat_y_test, flat_crf_y_pred)
crf_macro_f1 = f1_score(flat_y_test, flat_crf_y_pred, average="macro")
crf_entity_f1 = entity_f1_score(y_test, crf_y_pred_seq)

print(f"CRF Accuracy: {crf_accuracy:.4f}")
print(f"CRF Macro-F1: {crf_macro_f1:.4f}")
print(f"CRF Entity-F1: {crf_entity_f1:.4f}")

Preparing features for CRF...
Training CRF Model...
Evaluating CRF...
CRF Accuracy: 0.8372
CRF Macro-F1: 0.8106
CRF Entity-F1: 0.7292


## Public Vietnamese ELECTRA NER Model

### Token Alignment

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Mô hình công khai trên Hugging Face, đã fine-tune cho NER tiếng Việt.
model_checkpoint = "NlpHUST/ner-vietnamese-electra-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
public_model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)
public_model.eval()

# WikiANN dùng PER/ORG/LOC; mô hình NlpHUST dùng PERSON/ORGANIZATION/LOCATION.
# Ta chuẩn hóa nhãn để có thể so sánh trực tiếp với nhãn của WikiANN.
model_label_to_wikiann = {
    "B-PERSON": "B-PER",
    "I-PERSON": "I-PER",
    "B-ORGANIZATION": "B-ORG",
    "I-ORGANIZATION": "I-ORG",
    "B-LOCATION": "B-LOC",
    "I-LOCATION": "I-LOC",
    "B-MISCELLANEOUS": "O",
    "I-MISCELLANEOUS": "O",
    "O": "O",
}


def normalize_model_label(label):
    return model_label_to_wikiann.get(label, "O")


# Transformer tách từ thành subword, nên ta chỉ lấy nhãn ở subword đầu tiên của mỗi token gốc.
def predict_tags_for_tokens(tokens):
    inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )

    with torch.no_grad():
        logits = public_model(**inputs).logits

    predictions = torch.argmax(logits, dim=-1)[0].tolist()
    word_ids = inputs.word_ids(batch_index=0)

    tags = []
    previous_word_idx = None
    for pred_id, word_idx in zip(predictions, word_ids):
        if word_idx is None or word_idx == previous_word_idx:
            previous_word_idx = word_idx
            continue

        raw_label = public_model.config.id2label[pred_id]
        tags.append(normalize_model_label(raw_label))
        previous_word_idx = word_idx

    return tags

print("Token alignment complete!")

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

I0000 00:00:1779810461.532602 1165269 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779810461.886717 1165269 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779810463.511033 1165269 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


model.safetensors:   0%|          | 0.00/532M [00:00<?, ?B/s]

Token alignment complete!


### Evaluating the Public Model

In [7]:
print("Evaluating public Vietnamese ELECTRA NER model...")

electra_y_true_seq = []
electra_y_pred_seq = []
electra_y_true = []
electra_y_pred = []

for example in test_data:
    tokens = example["tokens"]
    true_tags = [label_names[tag] for tag in example["ner_tags"]]
    pred_tags = predict_tags_for_tokens(tokens)

    # Nếu câu bị cắt ở max_length, chỉ đánh giá phần còn giữ lại.
    true_tags = true_tags[:len(pred_tags)]

    electra_y_true_seq.append(true_tags)
    electra_y_pred_seq.append(pred_tags)
    electra_y_true.extend(true_tags)
    electra_y_pred.extend(pred_tags)

electra_accuracy = accuracy_score(electra_y_true, electra_y_pred)
electra_macro_f1 = f1_score(electra_y_true, electra_y_pred, average="macro")
electra_entity_f1 = entity_f1_score(electra_y_true_seq, electra_y_pred_seq)

print(f"Vietnamese ELECTRA Accuracy: {electra_accuracy:.4f}")
print(f"Vietnamese ELECTRA Macro-F1: {electra_macro_f1:.4f}")
print(f"Vietnamese ELECTRA Entity-F1: {electra_entity_f1:.4f}")

sample_sentence = "Đại học Quốc gia Thành phố Hồ Chí Minh hợp tác với Google tại Việt Nam."
sample_tokens = sample_sentence.split()
sample_predictions = predict_tags_for_tokens(sample_tokens)

print("\nVí dụ dự đoán:")
for token, tag in zip(sample_tokens, sample_predictions):
    print(f"{token:15s} -> {tag}")

Evaluating public Vietnamese ELECTRA NER model...
Vietnamese ELECTRA Accuracy: 0.6136
Vietnamese ELECTRA Macro-F1: 0.5429
Vietnamese ELECTRA Entity-F1: 0.4242

Ví dụ dự đoán:
Đại             -> B-ORG
học             -> I-ORG
Quốc            -> I-ORG
gia             -> I-ORG
Thành           -> I-ORG
phố             -> I-ORG
Hồ              -> I-ORG
Chí             -> I-ORG
Minh            -> I-ORG
hợp             -> O
tác             -> O
với             -> O
Google          -> B-ORG
tại             -> O
Việt            -> B-LOC
Nam.            -> I-LOC


# Results

In [8]:
# Create a summary DataFrame
summary_data = {
    "Model": ["Conditional Random Field (CRF)", "NlpHUST Vietnamese ELECTRA NER"],
    "Accuracy": [crf_accuracy, electra_accuracy],
    "Macro-F1 Score": [crf_macro_f1, electra_macro_f1],
    "Entity-F1 Score": [crf_entity_f1, electra_entity_f1],
}

summary_df = pd.DataFrame(summary_data)
summary_df.set_index("Model", inplace=True)

print("================ MODEL COMPARISON SUMMARY ================")
display(summary_df)

print("\n" + "="*58)
print("Detailed Classification Report for Vietnamese ELECTRA:")
print("="*58)
print(classification_report(electra_y_true, electra_y_pred, labels=label_names, zero_division=0))

================ MODEL COMPARISON SUMMARY ================


,Accuracy,Macro-F1 Score,Entity-F1 Score
Model,,,
Conditional Random Field (CRF),0.837202,0.810558,0.729204
NlpHUST Vietnamese ELECTRA NER,0.613622,0.542868,0.424242



Detailed Classification Report for Vietnamese ELECTRA:
              precision    recall  f1-score   support

           O       0.56      0.98      0.71      1190
       B-PER       0.70      0.81      0.75       188
       I-PER       0.81      0.61      0.70       378
       B-ORG       0.78      0.27      0.40       215
       I-ORG       0.85      0.30      0.45       740
       B-LOC       0.40      0.46      0.43       166
       I-LOC       0.63      0.26      0.37       397

    accuracy                           0.61      3274
   macro avg       0.68      0.53      0.54      3274
weighted avg       0.68      0.61      0.58      3274



### Nhận xét
CRF cho kết quả tốt hơn trong demo này vì học trực tiếp trên WikiANN tiếng Việt và tận dụng quan hệ nhãn liền kề, trong khi mô hình Vietnamese ELECTRA là mô hình công khai huấn luyện trên dữ liệu khác nên có thể lệch miền dữ liệu.